# 00. 데이터 크롤링 (네이버 카페 - dogpalza)

반려동물 보호자 커뮤니티(네이버 카페 dogpalza)에서 **3가지 맥락**의 게시글을 Selenium으로 수집합니다.

| 맥락 | 키워드 | 목표 |
|------|--------|------|
| 외출/산책 | 산책, 외출, 여행, 캐리어, 케이지, 동반, 펫택시 등 | 외출 시 보호자 페인포인트 |
| 건강/케어 | 건강, 케어, 병원, 수술, 영양제, 재활 등 | 건강관리 관련 니즈 |
| 하우스/환경 | 하우스, 냄새, 환기, 청결, 온도, 청소 등 | 주거환경 관련 니즈 |

## 1. 공통 설정 및 함수 정의

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
import pandas as pd
import time
from tqdm import tqdm

### 본문 크롤링 함수

각 맥락별로 href를 수집한 뒤, 아래 함수로 본문과 제목을 크롤링합니다.  
네이버 카페는 iframe 구조이므로 `switch_to.frame('cafe_main')` 전환이 필요합니다.

In [ ]:
def crawl_urls(urls):
    """네이버 카페 게시글 본문 크롤링 (iframe 전환 포함)"""
    titles = []
    contents = []
    
    for url in tqdm(urls, desc='크롤링 진행중', unit='페이지'):
        try:
            driver.get(url)
            time.sleep(1.8)

            driver.switch_to.default_content()
            driver.switch_to.frame('cafe_main')

            title = driver.find_element(By.CLASS_NAME, 'title_text').text
            content = driver.find_element(By.CLASS_NAME, 'se-main-container').text

            titles.append(title)
            contents.append(content)

        except Exception as e:
            print(f"[FAIL] {url} -> {e}")
            titles.append(None)
            contents.append(None)
    
    return titles, contents

## 2. 외출/산책 맥락 크롤링

**키워드**: 산책, 외출, 여행, 돌봄, 위탁, 이동장, 캐리어, 켄넬, 케이지, 동반, 펫택시, 호텔, 펫티켓, 에티켓, 가방, 지하철, 자동차, 차, 기차

### 2-1. 카페 접속 및 검색

In [ ]:
# 카페 접속
url = 'https://cafe.naver.com/dogpalza'
driver = wb.Chrome()
driver.get(url)

# 쫑알쫑알 게시판 클릭
driver.find_element(By.XPATH, '//*[@id="menuLink40"]').click()

# 50개 보기 설정
driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.BoardTopOption > div > div.FormSelectBox > div > button').click()
driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.BoardTopOption > div > div.FormSelectBox > div.select_option > ul > li:nth-child(7) > button').click()

# 검색어 입력
search_box = driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.SearchBoxLayout.type_bottom > div.SearchBox > div.search_input_area > input')
keywords = '여행 | 산책 | 외출 | 돌봄 | 위탁 | 이동장 | 캐리어 | 켄넬 | 케이지 | 동반 | 펫택시 | 호텔 | 펫티켓 | 에티켓 | 가방 | 지하철 | 자동차 | 차 | 기차'
search_box.click()
search_box.send_keys(keywords + Keys.ENTER)

### 2-2. 게시글 href 수집

In [ ]:
# 외출 맥락 - 게시글 URL 수집
f_url = []
driver = wb.Chrome()

for i in range(1, 100):
    url = f'https://cafe.naver.com/f-e/cafes/10258021/menus/40?viewType=L&page={i}&size=50&ta=ARTICLE_COMMENT&q=%EC%97%AC%ED%96%89+%7C+%EC%82%B0%EC%B1%85+%7C+%EC%99%B8%EC%B6%9C'
    driver.get(url)
    driver.implicitly_wait(10)
    aTags = driver.find_elements(By.CSS_SELECTOR, '.article')
    for tag in aTags:
        href = tag.get_attribute('href')
        f_url.append(href)

driver.quit()
print(f"외출 맥락 href 수집 완료: {len(f_url)}건")

### 2-3. 본문 크롤링 및 저장

In [ ]:
# 로그인 후 진행 필요
driver = wb.Chrome()
# → 브라우저에서 수동 로그인 후 아래 실행

titles, contents = crawl_urls(f_url)

df_외출맥락 = pd.DataFrame({'title': titles, 'content': contents})
df_외출맥락 = df_외출맥락.dropna()
df_외출맥락.to_csv('df_외출맥락.csv', index=False, encoding='utf-8-sig')
print(f"외출 맥락 저장 완료: {len(df_외출맥락)}건")

## 3. 건강/케어 맥락 크롤링

**키워드**: 건강, 케어, 관리, 병원, 진료, 수술, 약, 처방, 영양제, 하네스, 넥카라, 재활

### 3-1. 카페 접속 및 검색

In [ ]:
# 카페 접속
url = 'https://cafe.naver.com/dogpalza'
driver = wb.Chrome()
driver.get(url)

# 쫑알쫑알 게시판 클릭
driver.find_element(By.XPATH, '//*[@id="menuLink40"]').click()

# 50개 보기 설정
driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.BoardTopOption > div > div.FormSelectBox > div > button').click()
driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.BoardTopOption > div > div.FormSelectBox > div.select_option > ul > li:nth-child(7) > button').click()

# 검색어 입력
search_box = driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.SearchBoxLayout.type_bottom > div.SearchBox > div.search_input_area > input')
keywords = '건강 | 관리 | 케어 | 병원 | 약 | 수술 | 처방 | 영양제 | 하네스 | 넥카라 | 재활 | 진료'
search_box.click()
search_box.send_keys(keywords + Keys.ENTER)

### 3-2. 게시글 href 수집

In [ ]:
# 건강/케어 맥락 - 게시글 URL 수집
f_url = []
driver = wb.Chrome()

for i in range(1, 400):
    url = f'https://cafe.naver.com/f-e/cafes/10258021/menus/40?viewType=L&page={i}&size=50&ta=SUBJECT&q=%EA%B1%B4%EA%B0%95+%7C+%EA%B4%80%EB%A6%AC+%7C+%EC%BC%80%EC%96%B4+%7C+%EB%B3%91%EC%9B%90+%7C+%EC%95%BD+%7C+%EC%88%98%EC%88%A0+%7C+%EC%B2%98%EB%B0%A9+%7C+%EC%98%81%EC%96%91%EC%A0%9C+%7C+%ED%95%98%EB%84%A4%EC%8A%A4+%7C+%EB%84%A5%EC%B9%B4%EB%9D%BC+%7C+%EC%9E%AC%ED%99%9C+%7C+%EC%A7%84%EB%A3%8C'
    driver.get(url)
    driver.implicitly_wait(10)
    aTags = driver.find_elements(By.CSS_SELECTOR, '.article')
    for tag in aTags:
        href = tag.get_attribute('href')
        f_url.append(href)

driver.quit()
print(f"건강/케어 맥락 href 수집 완료: {len(f_url)}건")

# href 저장 (중간 백업)
df_건강 = pd.DataFrame(f_url, columns=['url'])
df_건강.to_csv('df_건강케어_href.csv', index=False)

### 3-3. 본문 크롤링 및 저장

In [ ]:
# 로그인 후 진행 필요
driver = wb.Chrome()
# → 브라우저에서 수동 로그인 후 아래 실행

titles, contents = crawl_urls(f_url)

df_건강맥락 = pd.DataFrame({'title': titles, 'content': contents})
df_건강맥락 = df_건강맥락.dropna()
df_건강맥락.to_csv('df_건강맥락.csv', index=False, encoding='utf-8-sig')
print(f"건강/케어 맥락 저장 완료: {len(df_건강맥락)}건")

## 4. 하우스/환경 맥락 크롤링

**키워드**: 하우스, 하우징, 하우스훈련, 냄새, 환기, 청결, 습도, 온도, 청소, 개집, 강아지집, 애견하우스

### 4-1. 카페 접속 및 검색

In [ ]:
# 카페 접속
url = 'https://cafe.naver.com/dogpalza'
driver = wb.Chrome()
driver.get(url)

# 쫑알쫑알 게시판 클릭
driver.find_element(By.XPATH, '//*[@id="menuLink40"]').click()

# 50개 보기 설정
driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.BoardTopOption > div > div.FormSelectBox > div > button').click()
driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.BoardTopOption > div > div.FormSelectBox > div.select_option > ul > li:nth-child(7) > button').click()

# 검색어 입력
search_box = driver.find_element(By.CSS_SELECTOR, '#cafe_content > div.SearchBoxLayout.type_bottom > div.SearchBox > div.search_input_area > input')
keywords = '하우스 | 하우징 | 하우스훈련 | 냄새 | 환기 | 청결 | 습도 | 온도 | 청소 | 개집 | 강아지집 | 애견하우스'
search_box.click()
search_box.send_keys(keywords + Keys.ENTER)

### 4-2. 게시글 href 수집

In [ ]:
# 하우스/환경 맥락 - 게시글 URL 수집
f_url = []
driver = wb.Chrome()

for i in range(1, 100):
    url = f'https://cafe.naver.com/f-e/cafes/10258021/menus/40?viewType=L&page={i}&size=50&ta=SUBJECT&q=%ED%95%98%EC%9A%B0%EC%8A%A4+%7C+%ED%95%98%EC%9A%B0%EC%A7%95+%7C+%ED%95%98%EC%9A%B0%EC%8A%A4%ED%9B%88%EB%A0%A8+%7C+%EB%83%84%EC%83%88+%7C+%ED%99%98%EA%B8%B0+%7C+%EC%B2%AD%EA%B2%B0+%7C+%EC%8A%B5%EB%8F%84+%7C+%EC%98%A8%EB%8F%84+%7C+%EC%B2%AD%EC%86%8C+%7C+%EA%B0%9C%EC%A7%91+%7C+%EA%B0%95%EC%95%84%EC%A7%80%EC%A7%91+%7C+%EC%95%A0%EA%B2%AC%ED%95%98%EC%9A%B0%EC%8A%A4'
    driver.get(url)
    driver.implicitly_wait(10)
    aTags = driver.find_elements(By.CSS_SELECTOR, '.article')
    for tag in aTags:
        href = tag.get_attribute('href')
        f_url.append(href)

driver.quit()
print(f"하우스/환경 맥락 href 수집 완료: {len(f_url)}건")

# href 저장 (중간 백업)
df_하우스 = pd.DataFrame(f_url, columns=['url'])
df_하우스.to_csv('df_하우스_href.csv', index=False)

### 4-3. 본문 크롤링 및 저장

In [ ]:
# 로그인 후 진행 필요
driver = wb.Chrome()
# → 브라우저에서 수동 로그인 후 아래 실행

titles, contents = crawl_urls(f_url)

df_하우스맥락 = pd.DataFrame({'title': titles, 'content': contents})
df_하우스맥락 = df_하우스맥락.dropna()
df_하우스맥락.to_csv('df_하우스맥락.csv', index=False, encoding='utf-8-sig')
print(f"하우스/환경 맥락 저장 완료: {len(df_하우스맥락)}건")